# RL4CRN tutorial notebook: Oscillator Frequency Task (CVODE)

This notebook shows an end-to-end **RL4CRN** workflow:

1. **Import utilities** and choose a **configuration preset**
2. **Build a template IO-CRN** (species, input/output mapping, solver settings)
3. **Build a reaction library** (MAK library in this example)
4. **Define a task** (here: *oscillator frequency* objective)
5. **Configure training**
6. **Create a session + trainer** and run training
7. **Inspect the best CRN** from the Hall of Fame

> Tip: If you ever see shape-related errors (e.g., ragged arrays), print and validate  
> `template_crn.num_inputs` vs `len(u_list[0])` to ensure input dimension consistency.


In [ ]:
import os, sys, numpy as np

print("Python:", sys.version.split()[0])
print("CWD:", os.getcwd())


## 1) Import RL4CRN helpers

We use the user-facing interface utilities from `RL4CRN.utils.input_interface`:

- `Configurator`: provides presets and override helpers
- `make_task`: builds a `TaskSpec` with reward function and input scenarios
- `make_session_and_trainer`: wires up environments, interfaces, policy, and agent
- `print_task_summary`: quick diagnostic summary for the created task


In [ ]:
from RL4CRN.utils.input_interface import (
    Configurator,
    make_task,
    make_session_and_trainer,
    print_task_summary,
)


## 2) Build a template IO/CRN

A **template IO-CRN** defines:
- the **species** in the model
- how **inputs** enter the system (via `input_map`)
- how **dilution** (if any) is applied
- which species is treated as the **output**
- the **ODE solver** and its tolerances

Here we use the convenience builder `build_simple_IOCRN`.


In [ ]:
from RL4CRN.utils.crn_builders import build_simple_IOCRN

# choose preset
cfg = Configurator.preset("paper")

# select simulator and set tolerances
cfg.solver.algorithm = "CVODE"
cfg.solver.rtol = 1e-10
cfg.solver.atol = 1e-10

# build template IO/CRN
species_labels = ['X_1', 'X_2', 'X_3']
crn, species_labels = build_simple_IOCRN(
    species=species_labels,
    production_input_map={"X_1": "u_1"},
    dilution_map={},
    output_species="X_3",
    solver=cfg.solver,
)

print("Template CRN built.")
print(" - num_inputs:", crn.num_inputs)
print(" - num_species:", len(species_labels))
print(" - species:", species_labels)


## 3) Build the reaction library (MAK)

RL4CRN typically proposes reactions from a **library**.  
This example uses a MAK library of given order (here `order=2`).

The builder returns a tuple `(library, M, K, masks)` that the training pipeline needs.


In [ ]:
from RL4CRN.utils.library_builders import build_MAK_library

# library components
library_components = build_MAK_library(crn, species_labels, order=2)

library, M, K, masks = library_components
print("Library built.")
print(" - M (num reactions in library):", M)
print(" - K (num parameters in library):", K)


## 4) Define the task: oscillator frequency

We define an **oscillation frequency matching** task.

- `kind="oscillator_freq"` selects the reward function based on oscillation frequency
- `u_values=[1/p for p in periods]` creates a grid over candidate frequencies
- `n_inputs` must match `template_crn.num_inputs`
- `ic=("constant", 0.01)` sets initial concentrations
- `osc_w` weights different components of the oscillation error  
  (here: `[mean, frequency, damping, periodicity_index]`)

We also choose the simulation window `t_f` and resolution `n_t`.


In [ ]:
periods = [10, 15, 20]

task = make_task(
    template_crn=crn,
    library_components=library_components,
    kind="oscillator_freq",
    species_labels=species_labels,   # passed again, as the crns are automatically compacted
    n_inputs=1,                     
    u_values=[1/p for p in periods],
    ic=("constant", 0.01),
    osc_w=[0/10, 6/10, 1/10, 3/10],  # [mean, frequency, damping, periodicity index]
    t_f=100, n_t=1000
)

print_task_summary(task)

# --- Optional safety checks (recommended) ---
print("Sanity checks:")
print(" - template num_inputs:", crn.num_inputs)
print(" - first u shape:", np.asarray(task.u_list[0]).shape)
print(" - first u length:", len(task.u_list[0]))
assert len(task.u_list[0]) == crn.num_inputs, "Input dimension mismatch: u has wrong length!"


## 5) Training configuration

We tune:
- `max_added_reactions`: episode length (how many reactions the agent can add)
- `epochs`: training iterations
- `render_every`: print progress cadence
- `seed`: reproducibility


In [ ]:
# ---- Train config ----
cfg.train.max_added_reactions = 5
cfg.train.epochs = 5
cfg.train.render_every = 5
cfg.train.seed = 0


## 6) Inspect full configuration (optional)

`cfg.describe()` prints a nested configuration dictionary.


In [ ]:
cfg.describe()


## 7) Create session + trainer

This step wires together:
- parallel environments
- observer/tensorizer/actuator/stepper interfaces
- policy + agent
- the chosen task reward function

The returned objects:
- `session`: contains all built components
- `trainer`: runs rollout → reward eval → policy update loops


In [ ]:
session, trainer = make_session_and_trainer(cfg, task)


## 8) Train and save checkpoints

We run for `cfg.train.epochs` epochs and periodically save a checkpoint.


In [ ]:
checkpoint_path = "oscillator_freq_task_chkpt.pkl"
trainer.run(epochs=cfg.train.epochs, checkpoint_path=checkpoint_path)


## 9) Inspect the best CRN

The trainer keeps a **Hall of Fame** of good CRNs found during rollouts.
We can:
- plot the best CRN transient response (if supported)
- print the best loss
- report Hall of Fame size


In [ ]:
trainer.inspect_best(plot=True)

best = trainer.best_crn()
print("Hall of Fame size:", len(session.mult_env.hall_of_fame))
if best is not None:
    print("Best loss:", best.last_task_info.get("reward", None))
